# Sandboxed Agent Execution

A sandbox limits the damage that generated code or tool actions can cause by isolating files, processes, network access, and credentials.


## What to learn

- Process isolation limits CPU, memory, time, and child processes.
- Filesystem policy exposes only required paths with explicit read or write access.
- Network policy blocks or allowlists outbound destinations.
- Short-lived credentials reduce the impact of leakage.

## Decision rule

Assume model-generated commands may be wrong or hostile. Start with no access, grant the minimum required capability, keep secrets outside the sandbox, require approval for escalation, and destroy the environment after use.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
# Import the dependencies used by this example.
from deepagents import FilesystemPermission, create_deep_agent

permissions = [
    FilesystemPermission(
        operations=["read", "write"],
        paths=["/workspace/.env", "/workspace/**/secrets/**"],
        mode="deny",
    ),
    FilesystemPermission(
        operations=["read", "write"],
        paths=["/workspace/**"],
        mode="allow",
    ),
    FilesystemPermission(
        operations=["read", "write"],
        paths=["/**"],
        mode="deny",
    ),
]

# Configure the framework object; this line prepares it but may not execute it yet.
agent = create_deep_agent(
    model="openai:gpt-5.6-sol",
    permissions=permissions,
    system_prompt="Work only in /workspace. Never request or expose credentials.",
)

# Built-in read_file/write_file/edit_file tools honor these rules.
# For arbitrary shell execution, pass a LangSmithSandbox, Modal, Daytona,
# or another SandboxBackendProtocol implementation as backend=.
# Execute the configured model or workflow with the input below.
result = agent.invoke({"messages": [{"role": "user", "content": "Create /workspace/report.md"}]})
result["messages"][-1].content


In [ ]:
"""Toy sandbox policy engine: a tool registry with capability descriptors
(filesystem paths, network hosts, secrets) plus a session policy. We evaluate
an agent-proposed action plan and emit ALLOW / DENY / NEEDS_APPROVAL verdicts
with an audit log. No API needed -- the "agent" is a scripted action list,
mirroring how Claude Code's proxy and srt-style configs make these calls."""

# Import the dependencies used by this example.
from dataclasses import dataclass
from fnmatch import fnmatch
import json

@dataclass(frozen=True)
# Define the data shape and small operations before running them.
class Capability:
    """Static descriptor of what a tool MAY touch (checked before policy)."""
    fs_read: tuple = ()
    fs_write: tuple = ()
    net_hosts: tuple = ()
    needs_secret: str = ""      # name only -- the sandbox never holds the value

REGISTRY = {
    "read_file":   Capability(fs_read=("/workspace/**",)),
    "write_file":  Capability(fs_write=("/workspace/**",)),
    "pip_install": Capability(net_hosts=("pypi.org", "*.pypi.org")),
    "http_get":    Capability(net_hosts=("*",)),  # deliberately broad tool
    "deploy":      Capability(net_hosts=("api.deploy.example",),
                              needs_secret="DEPLOY_TOKEN"),
}

POLICY = {
    "fs_read_deny":   ("/workspace/.env", "**/.ssh/**", "**/credentials*"),
    "fs_write_allow": ("/workspace/**",),
    "fs_write_deny":  ("**/.bashrc", "**/.git/config"),  # persistence vectors
    "net_allow":      ("pypi.org", "*.pypi.org", "api.github.com"),
    "approval_tools": ("deploy",),  # side-effectful: always gate on a human
}

def match(value, patterns):
    return any(fnmatch(value, p) for p in patterns)

def evaluate(action):
    tool, kind, target = action["tool"], action["kind"], action["target"]
    cap = REGISTRY.get(tool)
    if cap is None:
        return "DENY", "tool not in registry (default-deny)"
    # Layer 1: the tool's own declared capability (can it EVER do this?)
    declared = {"fs_read": cap.fs_read, "fs_write": cap.fs_write,
                "net": cap.net_hosts}.get(kind, ())
    if not match(target, declared):
        return "DENY", f"outside declared {kind} capability {list(declared)}"
    # Layer 2: session policy (may it do this HERE?)
    if kind == "fs_read":
        if match(target, POLICY["fs_read_deny"]):
            return "DENY", "matches secret-material deny pattern"
        return "ALLOW", "within capability; no deny pattern hit"
    if kind == "fs_write":
        if match(target, POLICY["fs_write_deny"]):
            return "DENY", "write to persistence vector (dotfile/config)"
        if not match(target, POLICY["fs_write_allow"]):
            return "DENY", "outside workspace write root"
        return "ALLOW", "inside workspace write root"
    if kind == "net":
        note = (f" [broker injects {cap.needs_secret} at proxy; "
                "secret never enters sandbox]" if cap.needs_secret else "")
        if tool in POLICY["approval_tools"]:
            return "NEEDS_APPROVAL", "side-effectful tool gated on human" + note
        if match(target, POLICY["net_allow"]):
            return "ALLOW", "host on egress allowlist" + note
        return "NEEDS_APPROVAL", "unknown host -> proxy asks user (Claude Code style)"
    return "DENY", f"unknown action kind {kind!r}"

AGENT_PLAN = [  # what a (possibly prompt-injected) agent proposes to do
    {"tool": "read_file",   "kind": "fs_read",  "target": "/workspace/src/app.py"},
    {"tool": "read_file",   "kind": "fs_read",  "target": "/workspace/.env"},
    {"tool": "read_file",   "kind": "fs_read",  "target": "/home/dev/.ssh/id_rsa"},
    {"tool": "write_file",  "kind": "fs_write", "target": "/workspace/out/report.md"},
    {"tool": "write_file",  "kind": "fs_write", "target": "/workspace/.git/config"},
    {"tool": "pip_install", "kind": "net",      "target": "files.pypi.org"},
    {"tool": "http_get",    "kind": "net",      "target": "api.github.com"},
    {"tool": "http_get",    "kind": "net",      "target": "paste.exfil.example"},
    {"tool": "deploy",      "kind": "net",      "target": "api.deploy.example"},
    {"tool": "rm_rf",       "kind": "fs_write", "target": "/"},
]

audit_log = []
for i, action in enumerate(AGENT_PLAN, 1):
    verdict, reason = evaluate(action)
    entry = {"step": i, **action, "verdict": verdict, "reason": reason}
    audit_log.append(entry)
    print(f"[{verdict:>14}] {action['tool']:<11} {action['kind']:<8} "
          f"{action['target']:<28} | {reason}")

counts = {}
for e in audit_log:
    counts[e["verdict"]] = counts.get(e["verdict"], 0) + 1
print("\nverdict summary:", counts)
print("\nsample audit record (ship these to append-only storage):")
print(json.dumps(audit_log[7], indent=2))


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
